In [1]:
import pandas as pd

df = pd.read_csv("D:/Hand_Gesture/data/processed_full_10k/labels.csv")

print(df.head())
print("total:", len(df))
print(df["label_name"].value_counts())
print(df["label"].value_counts().sort_index())

     idx  original_label original_label_name  label label_name  \
0  43685               2                fist      1       fist   
1  43686               2                fist      1       fist   
2  43687               2                fist      1       fist   
3  43688               2                fist      1       fist   
4  43689               2                fist      1       fist   

                                           crop_path  \
0  D:\Hand_Gesture\data\processed_full_10k\crops\...   
1  D:\Hand_Gesture\data\processed_full_10k\crops\...   
2  D:\Hand_Gesture\data\processed_full_10k\crops\...   
3  D:\Hand_Gesture\data\processed_full_10k\crops\...   
4  D:\Hand_Gesture\data\processed_full_10k\crops\...   

                                       landmark_path quality  
0  D:\Hand_Gesture\data\processed_full_10k\landma...      ok  
1  D:\Hand_Gesture\data\processed_full_10k\landma...      ok  
2  D:\Hand_Gesture\data\processed_full_10k\landma...      ok  
3  D:\Hand_Ges

In [2]:
train_df = (
    df.groupby("label", group_keys=False)
      .apply(lambda x: x.sample(frac=0.8, random_state=0))
)

val_df = df.drop(train_df.index)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("train:", len(train_df))
print("val:", len(val_df))

print("\nTrain class count:")
print(train_df["label_name"].value_counts())

print("\nVal class count:")
print(val_df["label_name"].value_counts())

train: 7366
val: 1841

Train class count:
label_name
N_A     3728
palm     755
ok       750
one      722
fist     706
like     705
Name: count, dtype: int64

Val class count:
label_name
N_A     932
palm    189
ok      187
one     180
fist    177
like    176
Name: count, dtype: int64


C:\Users\user\AppData\Local\Temp\ipykernel_48724\2640640765.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(frac=0.8, random_state=0))


In [3]:
import sys
!{sys.executable} -m pip install torch torchvision
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(10),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


class GestureImageDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img = Image.open(row["crop_path"]).convert("RGB")
        label = int(row["label"])

        if self.transform is not None:
            img = self.transform(img)

        return img, label


train_dataset = GestureImageDataset(train_df, transform=train_transform)
val_dataset = GestureImageDataset(val_df, transform=val_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False
)

imgs, labels = next(iter(train_loader))
print(imgs.shape)
print(labels)

torch.Size([16, 3, 224, 224])
tensor([0, 0, 1, 5, 1, 3, 4, 4, 4, 1, 1, 3, 3, 2, 3, 0])


In [4]:
import torch
import torch.nn as nn
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

weights = MobileNet_V3_Small_Weights.IMAGENET1K_V1
model = mobilenet_v3_small(weights=weights)

# MobileNetV3-small 原本是 ImageNet 1000 類
# 改成作業需要的 6 類
model.classifier[3] = nn.Linear(model.classifier[3].in_features, 6)

model = model.to(device)

device: cuda


In [5]:
import torch

print("torch version:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("torch cuda version:", torch.version.cuda)
print("device count:", torch.cuda.device_count())

torch version: 2.8.0+cu129
cuda available: True
torch cuda version: 12.9
device count: 1


# Image only

In [6]:
import sys
!{sys.executable} -m pip install -U tqdm ipywidgets
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm.notebook import tqdm
from pathlib import Path

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

save_path = Path("D:/Hand_Gesture/model/mobilenetv3_small_best.pth")
save_path.parent.mkdir(exist_ok=True)

def train_one_epoch(model, loader):
    model.train()

    total_loss = 0
    correct = 0
    total = 0

    pbar = tqdm(loader, desc="Training", leave=False)

    for imgs, labels in pbar:
        imgs = imgs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        logits = model(imgs)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * imgs.size(0)

        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        current_loss = total_loss / total
        current_acc = correct / total

        pbar.set_postfix({
            "loss": f"{current_loss:.4f}",
            "acc": f"{current_acc:.4f}"
        })

    return total_loss / total, correct / total


def evaluate(model, loader):
    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    pbar = tqdm(loader, desc="Validation", leave=False)

    with torch.no_grad():
        for imgs, labels in pbar:
            imgs = imgs.to(device)
            labels = labels.to(device)

            logits = model(imgs)
            loss = criterion(logits, labels)

            total_loss += loss.item() * imgs.size(0)

            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            current_loss = total_loss / total
            current_acc = correct / total

            pbar.set_postfix({
                "loss": f"{current_loss:.4f}",
                "acc": f"{current_acc:.4f}"
            })

    return total_loss / total, correct / total


best_val_acc = 0.0
num_epochs = 15

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch + 1}/{num_epochs}")

    train_loss, train_acc = train_one_epoch(model, train_loader)
    val_loss, val_acc = evaluate(model, val_loader)

    print(
        f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f}, "
        f"val_loss={val_loss:.4f}, val_acc={val_acc:.4f}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), save_path)
        print(f"Saved best model: val_acc={best_val_acc:.4f}")

print("\nBest val acc:", best_val_acc)
print("Best model saved to:", save_path)


Epoch 1/15


Training:   0%|          | 0/461 [00:00<?, ?it/s]

Validation:   0%|          | 0/116 [00:00<?, ?it/s]

train_loss=0.7027, train_acc=0.7452, val_loss=0.3133, val_acc=0.9011
Saved best model: val_acc=0.9011

Epoch 2/15


Training:   0%|          | 0/461 [00:00<?, ?it/s]

...

Validation:   0%|          | 0/116 [00:00<?, ?it/s]

train_loss=0.0416, train_acc=0.9862, val_loss=0.2895, val_acc=0.9316

Epoch 15/15


Training:   0%|          | 0/461 [00:00<?, ?it/s]

Validation:   0%|          | 0/116 [00:00<?, ?it/s]

train_loss=0.0524, train_acc=0.9825, val_loss=0.3072, val_acc=0.9321

Best val acc: 0.9429657794676806
Best model saved to: D:\Hand_Gesture\model\mobilenetv3_small_best.pth


In [7]:
from pathlib import Path

model_dir = Path("D:/Hand_Gesture/model")
model_dir.mkdir(exist_ok=True)

torch.save(model.state_dict(), model_dir / "mobilenetv3_small_best.pth")

print("saved to:", model_dir / "mobilenetv3_small_best.pth")

saved to: D:\Hand_Gesture\model\mobilenetv3_small_best.pth


In [8]:
from pathlib import Path

model_path = Path("D:/Hand_Gesture/model/mobilenetv3_small_best.pth")

file_size_mb = model_path.stat().st_size / (1024 ** 2)

print("Model file:", model_path)
print("Saved .pth size:", f"{file_size_mb:.2f} MB")

num_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

param_size_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
buffer_size_bytes = sum(b.numel() * b.element_size() for b in model.buffers())

model_memory_mb = (param_size_bytes + buffer_size_bytes) / (1024 ** 2)

print("Total parameters:", f"{num_params / 1e6:.3f} M")
print("Trainable parameters:", f"{trainable_params / 1e6:.3f} M")
print("Parameter + buffer size:", f"{model_memory_mb:.2f} MB")

Model file: D:\Hand_Gesture\model\mobilenetv3_small_best.pth
Saved .pth size: 5.94 MB
Total parameters: 1.524 M
Trainable parameters: 1.524 M
Parameter + buffer size: 5.86 MB


# 1.a : output feature vectors 

In [6]:
import torch
import torch.nn as nn
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights


class MobileNetV3SmallFeatureExtractor(nn.Module):
    def __init__(self, output_dim=128):
        super().__init__()

        weights = MobileNet_V3_Small_Weights.IMAGENET1K_V1
        backbone = mobilenet_v3_small(weights=weights)

        # 保留 MobileNetV3-small 的 CNN feature extractor
        self.features = backbone.features
        self.avgpool = backbone.avgpool

        # MobileNetV3-small avgpool 後通常是 576 維
        self.projector = nn.Sequential(
            nn.Linear(576, output_dim),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.projector(x)
        return x

In [7]:
image_encoder = MobileNetV3SmallFeatureExtractor(output_dim=128)
image_encoder = image_encoder.to(device)

In [9]:
imgs, labels = next(iter(train_loader))

imgs = imgs.to(device)

image_encoder.eval()

with torch.no_grad():
    features = image_encoder(imgs)

print("imgs shape:", imgs.shape)
print("labels shape:", labels.shape)
print("features shape:", features.shape)

imgs shape: torch.Size([32, 3, 224, 224])
labels shape: torch.Size([32])
features shape: torch.Size([32, 128])


## Validation

In [9]:
import torch
import pandas as pd
import numpy as np

class_names = ["N_A", "fist", "like", "ok", "one", "palm"]

def evaluate_detailed(model, loader, criterion=None, threshold=None):
    model.eval()

    total_loss = 0
    total = 0

    all_labels = []
    all_preds = []
    all_confs = []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            labels = labels.to(device)

            logits = model(imgs)

            if criterion is not None:
                loss = criterion(logits, labels)
                total_loss += loss.item() * imgs.size(0)

            probs = torch.softmax(logits, dim=1)
            confs, preds = torch.max(probs, dim=1)

            if threshold is not None:
                preds = torch.where(confs < threshold, torch.zeros_like(preds), preds)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_confs.extend(confs.cpu().numpy())

            total += labels.size(0)

    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)
    all_confs = np.array(all_confs)

    acc = (all_labels == all_preds).mean()

    target_mask = all_labels != 0
    na_mask = all_labels == 0

    target_acc = (all_labels[target_mask] == all_preds[target_mask]).mean() if target_mask.sum() > 0 else 0
    na_false_trigger_rate = (all_preds[na_mask] != 0).mean() if na_mask.sum() > 0 else 0

    avg_loss = total_loss / total if criterion is not None else None

    cm = pd.crosstab(
        pd.Series(all_labels, name="True"),
        pd.Series(all_preds, name="Pred"),
        dropna=False
    )

    result_df = pd.DataFrame({
        "true": all_labels,
        "pred": all_preds,
        "confidence": all_confs,
        "true_name": [class_names[i] for i in all_labels],
        "pred_name": [class_names[i] for i in all_preds],
    })

    metrics = {
        "loss": avg_loss,
        "accuracy": acc,
        "target_accuracy": target_acc,
        "na_false_trigger_rate": na_false_trigger_rate,
    }

    return metrics, cm, result_df

In [10]:
metrics, cm, result_df = evaluate_detailed(
    model,
    val_loader,
    criterion=criterion,
    threshold=None
)

print(metrics)
display(cm)
display(result_df.head())

{'loss': 1.376908818880717, 'accuracy': np.float64(0.5128205128205128), 'target_accuracy': np.float64(0.05), 'na_false_trigger_rate': np.float64(0.0)}


Pred,0,1
True,,
0,19,0
1,3,1
2,4,0
3,4,0
4,4,0
5,4,0


,true,pred,confidence,true_name,pred_name
0,1,0,0.323304,fist,N_A
1,1,0,0.392466,fist,N_A
2,1,1,0.287906,fist,fist
3,1,0,0.307529,fist,N_A
4,2,0,0.401630,like,N_A


# Results

train_loss=0.0524, train_acc=0.9825, val_loss=0.3072, val_acc=0.9321

Best val acc: 0.9429657794676806
Best model saved to: D:\Hand_Gesture\model\mobilenetv3_small_best.pth